In [10]:
import os
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm

In [ ]:
import subprocess
print("Full Dataset Size:\n")
result = subprocess.run(['du', '-sb', '/kaggle/input/competitions/diabetic-retinopathy-detection'],capture_output=True, text=True)
total_bytes = int(result.stdout.split()[0])
size_gb  = total_bytes / 1_000_000_000      # dec
size_gib = total_bytes / (1024 ** 3)        # binary
print(f"Decimal (GB): {size_gb:.2f} GB")
print(f"Binary  (GiB): {size_gib:.2f} GiB")

In [ ]:
import os
input_dir = '/kaggle/input/competitions/diabetic-retinopathy-detection'
print("Dataset files directory:\n")
for item in sorted(os.listdir(input_dir)):
    print(f"  {item}")

In [ ]:
# import os
# import cv2
# import math
# import shutil
# import subprocess
# from pathlib import Path
# from tqdm import tqdm

# FIRST_PART = "/kaggle/input/competitions/diabetic-retinopathy-detection/train.zip.001"
# TEMP_DIR = "/kaggle/working/temp_extract"
# OUTPUT_DIR = "/kaggle/working/train_images_compressed"
# LISTING_TXT = "/kaggle/working/train_listing.txt"
# CHUNK_LIST = "/kaggle/working/current_chunk.txt"

# TARGET_SIZE = (224, 224)
# QUALITY = 75
# CHUNK_SIZE = 3000 #tried 500,1000,2000,3000

# Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)
# Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# def run_cmd(cmd):
#     result = subprocess.run(cmd, capture_output=True, text=True)
#     return result.returncode, result.stdout, result.stderr

# def clear_dir(path):
#     if os.path.exists(path):
#         shutil.rmtree(path)
#     os.makedirs(path, exist_ok=True)
# def show_disk():
#     code, out, err = run_cmd(["df", "-h", "/kaggle/working"])
#     print(out if out else err)

# def count_images(folder):
#     total = 0
#     for _, _, files in os.walk(folder):
#         total += sum(f.lower().endswith((".jpeg", ".jpg", ".png")) for f in files)
#     return total

# print("\n1) Reading archive listing...")
# code, out, err = run_cmd(["7z", "l", FIRST_PART])

# if code != 0:
#     print(" Could not archive list")
#     print(err)
#     raise SystemExit

# with open(LISTING_TXT, "w") as f:
#     f.write(out)

# print("Archive listing saved:", LISTING_TXT)

In [ ]:
import os
import cv2
import math
import shutil
import subprocess
from pathlib import Path
from tqdm import tqdm

FIRST_PART = "/kaggle/input/competitions/diabetic-retinopathy-detection/train.zip.001"# will use 7z for multiple train.zipFile
TEMP_DIR = "/kaggle/working/temp_extract"
OUTPUT_DIR = "/kaggle/working/train_images_compressed"
LISTING_TXT = "/kaggle/working/train_listing.txt"


TARGET_SIZE = (224, 224)
QUALITY = 75
CHUNK_SIZE = 3000 

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def run_cmd(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode, result.stdout, result.stderr

def clear_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def show_disk():
    code, out, err = run_cmd(["df", "-h", "/kaggle/working"])
    print(out if out else err)

def count_images(folder):
    total = 0
    for _, _, files in os.walk(folder):
        total += sum(f.lower().endswith((".jpeg", ".jpg", ".png")) for f in files)
    return total

# List archive -----------------------------------
# print("\n1) Reading archive listing...")
code, out, err = run_cmd(["7z", "l", FIRST_PART])

if code != 0:
    print("Cant archive list")
    print(err)
    raise SystemExit

with open(LISTING_TXT, "w") as f:
    f.write(out)

#print("✓ Archive listing saved to:", LISTING_TXT)

# Parse filenames------------------------------
image_files = []
for line in out.splitlines():
    line = line.strip()
    if not line:
        continue
    if " train/" in " " + line and line.lower().endswith((".jpeg", ".jpg", ".png")):
        filename = line.split()[-1]
        if filename.startswith("train/"):
            image_files.append(filename)

# removing duplicates
seen = set()
image_files = [x for x in image_files if not (x in seen or seen.add(x))]

print(f"Total image files found: {len(image_files)}")
print("Sample parsed files:", image_files[:5])

if len(image_files) == 0:
    print("No image Parsed!!")
    raise SystemExit

# Small extraction test--------------------------------------------------
print("\n2) Running small extraction test...")
clear_dir(TEMP_DIR)

test_files = image_files[:2]
print("Testing with:", test_files)

code, out, err = run_cmd([
    "7z", "x", FIRST_PART,
    f"-o{TEMP_DIR}",
    "-y",
    *test_files
])

print("7z test return code:", code)
if out:
    print(out[:1000])
if err:
    print(err[:500])

test_extracted = []
for root, dirs, files in os.walk(TEMP_DIR):
    for f in files:
        if f.lower().endswith((".jpeg", ".jpg", ".png")):
            test_extracted.append(os.path.join(root, f))

print("Test Extracted Files:", len(test_extracted))
for p in test_extracted[:5]:
    print("  ", p)

if len(test_extracted) == 0:
    print("Test Extraction: 0 images.")
    raise SystemExit

# cleanup 
clear_dir(TEMP_DIR)

# Chunked processing--------------------------------------------------
num_chunks = math.ceil(len(image_files) / CHUNK_SIZE)
print(f"\n3) Starting chunked processing")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Total chunks: {num_chunks}")

print("\nInitial disk usage:")
show_disk()

total_processed = 0
total_failed = 0

for chunk_idx in range(num_chunks):
    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(image_files))
    chunk_files = image_files[start:end]

    print(f"Chunk {chunk_idx + 1}/{num_chunks} | files {start} to {end - 1}")
    clear_dir(TEMP_DIR)
    cmd = ["7z", "x", FIRST_PART, f"-o{TEMP_DIR}", "-y"] + chunk_files
    code, out, err = run_cmd(cmd)

    print("7z return code:", code)
    if code != 0:
        print("Extraction failed !!!")
        print(err[:1000])
        total_failed += len(chunk_files)
        continue

    extracted_paths = []
    for root, dirs, files in os.walk(TEMP_DIR):
        for f in files:
            if f.lower().endswith((".jpeg", ".jpg", ".png")):
                extracted_paths.append(os.path.join(root, f))

    print(f"Extracted files in chunk: {len(extracted_paths)}")

    if len(extracted_paths) == 0:
        print("No files extracted !!")
        continue

    processed_this_chunk = 0
    failed_this_chunk = 0

    for img_path in tqdm(extracted_paths, desc="Compressing"):
        try:
            img = cv2.imread(img_path)
            if img is None:
                failed_this_chunk += 1
                total_failed += 1
                continue

            img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)

            out_name = os.path.splitext(os.path.basename(img_path))[0] + ".jpg"
            out_path = os.path.join(OUTPUT_DIR, out_name)

            ok = cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, QUALITY])

            if ok and os.path.exists(out_path):
                processed_this_chunk += 1
                total_processed += 1
            else:
                failed_this_chunk += 1
                total_failed += 1

        except Exception as e:
            failed_this_chunk += 1
            total_failed += 1

    print(f"Processed this chunk: {processed_this_chunk}")
    print(f"Failed this chunk: {failed_this_chunk}")
    print(f"Current saved images in output: {count_images(OUTPUT_DIR)}")

    clear_dir(TEMP_DIR)

    print("Disk after cleanup:")
    show_disk()

#Final completion--------------------------------------------------

print("Chunks Completedd")
print(f"Total processed: {total_processed}")
print(f"Total failed: {total_failed}")
print(f"Output folder: {OUTPUT_DIR}")

final_count = count_images(OUTPUT_DIR)
print(f"\nFinal image count in output folder: {final_count}")

print("\nOutput size:")
os.system(f'du -sh "{OUTPUT_DIR}"')

# print("\nFirst 10 saved files:")
# if os.path.exists(OUTPUT_DIR):
#     saved = sorted(os.listdir(OUTPUT_DIR))
#     print(saved[:10])

# Zip the output 
ZIP_PATH = "/kaggle/working/train_images_compressed.zip"
print("\nCreating zip archive of output folder...")
shutil.make_archive("/kaggle/working/train_images_compressed", "zip", OUTPUT_DIR)
print("Created:", ZIP_PATH)
#print("Exists:", os.path.exists(ZIP_PATH))

In [4]:
# # Clear output folder
# import os

# def remove_folder_contents(folder):
#     for the_file in os.listdir(folder):
#         file_path = os.path.join(folder, the_file)
#         try:
#             if os.path.isfile(file_path):
#                 os.unlink(file_path)
#             elif os.path.isdir(file_path):
#                 remove_folder_contents(file_path)
#                 os.rmdir(file_path)
#         except Exception as e:
#             print(e)

# folder_path = '/kaggle/working/'